# Fixed propagation delay

This notebook demonstrates the same ideal, matched transmission line in the time and frequency domains. Circulax keeps the delay explicit rather than replacing it with a rational approximation:

$$b_1(t)=a\,a_2(t-\tau), \qquad b_2(t)=a\,a_1(t-\tau).$$

`TransmissionLine` expresses this directly in its component physics with `past = signals.at_delay(tau)`. In transient analysis this produces a literal time shift. In AC analysis it produces the corresponding phase factor $\exp(-j2\pi f\tau)$. The analyses below therefore test the same component physics rather than separate models.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import compile_circuit
from circulax.components.electronic import Resistor, SmoothPulse, TransmissionLine, VoltageSourceAC

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (8, 3.5), "axes.grid": True})

## Transient: watch the edge move

A smooth step drives a matched line. Because the load equals $Z_0$, there is no reflection and the output should be an attenuated copy of the input shifted by $\tau$. Constant DC prehistory means no artificial edge appears at startup.

In [ ]:
tau = 1.0e-9
z0 = 50.0
attenuation = 0.9
source_delay = 2.0e-9
rise_time = 0.8e-9

models = {
    "source": SmoothPulse,
    "line": TransmissionLine,
    "resistor": Resistor,
    "ground": lambda: 0,
}

transient_netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "VIN": {
            "component": "source",
            "settings": {"V": 1.0, "delay": source_delay, "tr": rise_time},
        },
        "TL": {
            "component": "line",
            "settings": {"tau": tau, "z0": z0, "attenuation": attenuation},
        },
        "RL": {"component": "resistor", "settings": {"R": z0}},
    },
    "connections": {
        "GND,p1": ("VIN,p2", "RL,p2"),
        "VIN,p1": "TL,p1",
        "TL,p2": "RL,p1",
    },
    "ports": {"input": "TL,p1", "output": "TL,p2"},
}

transient_circuit = compile_circuit(transient_netlist, models)

In [ ]:
sample_times = jnp.linspace(0.0, 6.0e-9, 601)
solution = transient_circuit.transient(
    t0=0.0,
    t1=float(sample_times[-1]),
    dt0=1.0e-11,
    saveat=sample_times,
    max_steps=4000,
    throw=True,
)

v_in = transient_circuit.port(solution.ys, "input")
v_out = transient_circuit.port(solution.ys, "output")
analytic_out = attenuation * jax.nn.sigmoid(
    10.0 * (sample_times - source_delay - tau) / rise_time
)
max_transient_error = float(jnp.max(jnp.abs(v_out - analytic_out)))
print(f"Maximum transient error: {max_transient_error:.2e}")
assert max_transient_error < 2e-4

In [ ]:
fig, ax = plt.subplots()
ax.plot(sample_times * 1e9, v_in, label="input", lw=2)
ax.plot(sample_times * 1e9, v_out, label="delayed output", lw=2)
ax.plot(sample_times * 1e9, analytic_out, "k--", label="analytic shift", lw=1.5)
ax.annotate(
    rf"$\tau={tau * 1e9:.1f}\,\mathrm{{ns}}$",
    xy=(source_delay * 1e9 + tau * 1e9 / 2, 0.5),
    ha="center",
)
ax.set(xlabel="Time (ns)", ylabel="Voltage (V)", title="A matched line delays the waveform without reshaping it")
ax.legend()
plt.show()

## AC: read the same delay from phase

For a matched line, $S_{21}=a\exp(-j2\pi f\tau)$. Its magnitude is constant and its unwrapped phase is linear, so the group delay is

$$-\frac{1}{2\pi}\frac{d\angle S_{21}}{df}=\tau.$$

In [ ]:
# Very large shunts register both external nodes while changing S by less than 1e-11.
ac_netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "TL": {
            "component": "line",
            "settings": {"tau": tau, "z0": z0, "attenuation": attenuation},
        },
        "R1": {"component": "resistor", "settings": {"R": 1e15}},
        "R2": {"component": "resistor", "settings": {"R": 1e15}},
    },
    "connections": {
        "GND,p1": ("R1,p2", "R2,p2"),
        "TL,p1": "R1,p1",
        "TL,p2": "R2,p1",
    },
    "ports": {"port1": "TL,p1", "port2": "TL,p2"},
}

ac_circuit = compile_circuit(ac_netlist, models)
frequencies = jnp.linspace(1e6, 2e9, 501)
scattering = ac_circuit.sp(ports=["port1", "port2"], freqs=frequencies, z0=z0)
s21 = scattering[:, 1, 0]
analytic_s21 = attenuation * jnp.exp(-2j * jnp.pi * frequencies * tau)
max_ac_error = float(jnp.max(jnp.abs(s21 - analytic_s21)))
print(f"Maximum AC error: {max_ac_error:.2e}")
assert max_ac_error < 1e-9

In [ ]:
phase = np.unwrap(np.angle(np.asarray(s21)))
group_delay = -np.gradient(phase, np.asarray(frequencies)) / (2 * np.pi)

fig, (ax_mag, ax_delay) = plt.subplots(1, 2, figsize=(10, 3.5))
ax_mag.plot(np.asarray(frequencies) / 1e9, 20 * np.log10(np.abs(np.asarray(s21))), lw=2)
ax_mag.axhline(20 * np.log10(attenuation), color="k", ls="--", label="analytic")
ax_mag.set(xlabel="Frequency (GHz)", ylabel=r"$|S_{21}|$ (dB)", title="Constant attenuation")
ax_mag.legend()

ax_delay.plot(np.asarray(frequencies) / 1e9, group_delay * 1e9, lw=2)
ax_delay.axhline(tau * 1e9, color="k", ls="--", label=rf"$\tau={tau * 1e9:.1f}$ ns")
ax_delay.set(xlabel="Frequency (GHz)", ylabel="Group delay (ns)", title="Delay recovered from phase")
ax_delay.legend()
plt.tight_layout()
plt.show()

## Harmonic balance: rotate each harmonic

HB represents a periodic waveform by harmonics of a fundamental frequency $f_0$. The solver applies the delay to harmonic $k$ as

$$X_k(t-\tau)=X_k(t)\exp(-j2\pi kf_0\tau).$$

Here a sinusoidal source drives the same matched line through a $50\,\Omega$ source resistance. We compare the fundamental output/input ratio with the same analytical factor used for AC.

In [ ]:
fundamental = 250e6
hb_netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "VS": {"component": "source_ac", "settings": {"V": 1.0, "freq": fundamental}},
        "RS": {"component": "resistor", "settings": {"R": z0}},
        "TL": {
            "component": "line",
            "settings": {"tau": tau, "z0": z0, "attenuation": attenuation},
        },
        "RL": {"component": "resistor", "settings": {"R": z0}},
    },
    "connections": {
        "GND,p1": ("VS,p2", "RL,p2"),
        "VS,p1": "RS,p1",
        "RS,p2": "TL,p1",
        "TL,p2": "RL,p1",
    },
    "ports": {"line_input": "TL,p1", "line_output": "TL,p2"},
}

hb_models = {**models, "source_ac": VoltageSourceAC}
hb_circuit = compile_circuit(hb_netlist, hb_models, backend="dense")
hb_time, hb_spectrum = hb_circuit.hb(
    freq=fundamental,
    harmonics=5,
    rtol=1e-9,
    atol=1e-9,
    max_steps=30,
)

input_spectrum = hb_circuit.port(hb_spectrum, "line_input")
output_spectrum = hb_circuit.port(hb_spectrum, "line_output")
hb_ratio = output_spectrum[1] / input_spectrum[1]
analytic_ratio = attenuation * jnp.exp(-2j * jnp.pi * fundamental * tau)
hb_error = float(jnp.abs(hb_ratio - analytic_ratio))
higher_harmonics = float(jnp.max(jnp.abs(output_spectrum[2:])))

print(f"HB fundamental ratio: {hb_ratio:.6f}")
print(f"Analytical ratio:      {analytic_ratio:.6f}")
print(f"Fundamental error:     {hb_error:.2e}")
print(f"Largest higher harmonic: {higher_harmonics:.2e}")
assert hb_error < 2e-7
assert higher_harmonics < 2e-8

In [ ]:
hb_times = np.arange(hb_time.shape[0]) / (hb_time.shape[0] * fundamental)
hb_input = np.asarray(hb_circuit.port(hb_time, "line_input"))
hb_output = np.asarray(hb_circuit.port(hb_time, "line_output"))
expected_hb_output = 0.5 * attenuation * np.sin(2 * np.pi * fundamental * (hb_times - tau))

fig, ax = plt.subplots()
ax.plot(hb_times * 1e9, hb_input, "o-", label="line input")
ax.plot(hb_times * 1e9, hb_output, "o-", label="HB delayed output")
ax.plot(hb_times * 1e9, expected_hb_output, "k--", label="analytic shift")
ax.set(xlabel="Time within one period (ns)", ylabel="Voltage (V)", title="HB periodic steady state")
ax.legend()
plt.show()

Transient, AC, and HB all recover the configured **1 ns** propagation delay. Because `TransmissionLine` requests one fixed delay through `signals.at_delay(tau)`, each solver infers and applies the appropriate time- or frequency-domain representation automatically.

## Fit a noisy transmission-line response

For a device known to be a transmission line, the usual starting point would be to fit a physical transmission-line model—its delay, characteristic impedance, loss, and dispersion as needed. This example uses delay-aware rational fitting for illustration: it gives us a known response against which to check noise handling, delay recovery, and circuit integration. It is not a recommendation to replace a suitable transmission-line model.

Now treat the same line as a measured device. We'll add complex Gaussian noise to its S-parameters, reserve every fifth frequency, and fit the remaining samples with and without delay de-embedding.

The noise has RMS magnitude 0.001 in each independent S entry. Real and imaginary parts each have standard deviation $0.001/\sqrt{2}$. We use the same noise in $S_{12}$ and $S_{21}$ to keep this synthetic dataset reciprocal; the two reflections have independent noise. The seed is fixed. A copy is available as a [Touchstone file](../fitting/data/noisy_transmission_line.s2p).

This uses Circulax's per-port de-embedding around conventional vector fitting. It isn't the more general multi-delay DVF algorithm discussed in the [fitting notebook](../fitting/vector_fitting.ipynb).


In [ ]:
from circulax.fitting import ModelCoefficients, ModelFitOptions, component_from_coefficients, fit_model

fit_frequencies = np.asarray(frequencies)
clean_s = np.zeros((len(fit_frequencies), 2, 2), dtype=complex)
clean_s[:, 0, 1] = clean_s[:, 1, 0] = attenuation * np.exp(-2j * np.pi * fit_frequencies * tau)
np.testing.assert_allclose(scattering, clean_s, atol=1e-9)

noise_rms = 0.001
rng = np.random.default_rng(2026)
noise = noise_rms / np.sqrt(2) * (
    rng.normal(size=(len(fit_frequencies), 3)) + 1j * rng.normal(size=(len(fit_frequencies), 3))
)
noisy_s = clean_s.copy()
noisy_s[:, 0, 0] += noise[:, 0]
noisy_s[:, 0, 1] += noise[:, 1]
noisy_s[:, 1, 0] += noise[:, 1]
noisy_s[:, 1, 1] += noise[:, 2]
train = np.arange(len(fit_frequencies)) % 5 != 0
holdout = ~train
print(f"Training samples: {train.sum()}, held-out samples: {holdout.sum()}")


### Fit and estimate the delay

The fitter receives the noisy training samples and an upper bound of 1.5 ns on the total transmission delay. It doesn't receive the true 1 ns value. Transmission determines only the sum of port delays, so automatic mode splits that estimate equally between the ports.

We allow reflection magnitudes up to five times the known noise RMS for the equal-split proposal. The error targets are also set above the noise floor: 0.3% normalized RMS and 0.005 maximum absolute S error. Trying to fit below the noise floor would spend poles on noise. Automatic mode compares a baseline with full- and half-delay candidates and keeps a passing delayed model only if it uses fewer poles.


In [ ]:
fit_limits = {"normalized_rmse": 0.003, "max_absolute_error": 0.005}
rational_fit = fit_model(
    noisy_s[train], fit_frequencies[train], z0=z0,
    options=ModelFitOptions(delay_mode="none", **fit_limits),
)
delayed_fit = fit_model(
    noisy_s[train], fit_frequencies[train], z0=z0,
    options=ModelFitOptions(
        delay_mode="auto", auto_max_delay=1.5e-9,
        auto_reflection_threshold=5 * noise_rms, **fit_limits,
    ),
)

estimated_delay = float(delayed_fit.port_delays.sum())
print(f"Estimated transmission delay: {estimated_delay * 1e9:.6f} ns (true: {tau * 1e9:.6f} ns)")
print("One-way port delays (ns):", delayed_fit.port_delays * 1e9)
print(f"Rational poles: {len(rational_fit.poles)} without delays, {len(delayed_fit.poles)} with delays")
for candidate in delayed_fit.metadata["auto_selection"]["candidates"]:
    print(candidate)
assert delayed_fit.metadata["auto_selection"]["selected"] == "delayed"
assert abs(estimated_delay - tau) < 0.005 * tau
assert len(delayed_fit.poles) < len(rational_fit.poles)


### Check samples the fitter didn't see

We compare both models against the noisy holdout samples and the clean response. The clean response is available because this is synthetic data; it isn't used to estimate the delay or choose a fit. The last column measures how well each fit recovers the line underneath the noise.


In [ ]:
def normalized_error(prediction, reference):
    return np.linalg.norm(prediction - reference) / np.linalg.norm(reference)

print(f"{'Model':<18} {'Poles':>6} {'Holdout NRMSE':>16} {'Holdout max':>14} {'Clean NRMSE':>14}")
for label, coefficients in [("Rational only", rational_fit), ("Explicit delays", delayed_fit)]:
    prediction = coefficients.evaluate(fit_frequencies[holdout])
    measured_error = normalized_error(prediction, noisy_s[holdout])
    maximum = np.max(np.abs(prediction - noisy_s[holdout]))
    clean_error = normalized_error(prediction, clean_s[holdout])
    print(f"{label:<18} {len(coefficients.poles):>6} {measured_error:>16.3e} {maximum:>14.3e} {clean_error:>14.3e}")
    assert measured_error < fit_limits["normalized_rmse"]
    assert maximum < fit_limits["max_absolute_error"]
assert normalized_error(delayed_fit.evaluate(fit_frequencies[holdout]), clean_s[holdout]) < 0.001

fit_s = delayed_fit.evaluate(fit_frequencies)
ghz = fit_frequencies / 1e9
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(ghz, 20 * np.log10(np.abs(noisy_s[:, 1, 0])), ".", ms=2, alpha=0.5, label="noisy samples")
axes[0].plot(ghz, 20 * np.log10(np.abs(fit_s[:, 1, 0])), label="delay-aware fit")
axes[0].axhline(20 * np.log10(attenuation), color="k", ls="--", label="clean line")
axes[0].set(xlabel="Frequency (GHz)", ylabel="Transmission (dB)")
axes[0].legend()
axes[1].plot(ghz, np.unwrap(np.angle(noisy_s[:, 1, 0])), ".", ms=2, alpha=0.5)
axes[1].plot(ghz, np.unwrap(np.angle(fit_s[:, 1, 0])), label="delay-aware fit")
axes[1].plot(ghz, -2 * np.pi * fit_frequencies * tau, "k--", label="clean line")
axes[1].set(xlabel="Frequency (GHz)", ylabel="Transmission phase (rad)")
axes[1].legend()
axes[2].semilogy(ghz, np.abs(noisy_s[:, 1, 0] - clean_s[:, 1, 0]), ".", ms=2, alpha=0.5, label="noise")
axes[2].semilogy(ghz, np.abs(fit_s[:, 1, 0] - clean_s[:, 1, 0]), label="fit error")
axes[2].set(xlabel="Frequency (GHz)", ylabel="Absolute transmission error")
axes[2].legend()
plt.tight_layout()
plt.show()


### Load the fitted line into a transient circuit

Save and reload the coefficients, then replace the original transmission line in the pulse circuit with the fitted model. The recovered model includes its delays, so this uses the ordinary transient solver.

The sweep covers 1 MHz–2 GHz. Comparing a pulse with the clean line also tests extrapolation outside that band; we can check it here because we know the generating model. Agreement on this example doesn't establish extrapolation accuracy for an unknown measured device.


In [ ]:
import copy
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as directory:
    coefficient_path = Path(directory) / "noisy_line_fit.npz"
    delayed_fit.save(coefficient_path)
    restored_fit = ModelCoefficients.load(coefficient_path)
    fitted_line = component_from_coefficients(restored_fit, name="NoisyLineFit")
np.testing.assert_allclose(restored_fit.evaluate(fit_frequencies), fit_s)

fitted_netlist = copy.deepcopy(transient_netlist)
fitted_netlist["instances"]["TL"] = {"component": "fitted_line"}
fitted_circuit = compile_circuit(fitted_netlist, {**models, "fitted_line": fitted_line})
fitted_solution = fitted_circuit.transient(
    t0=0.0, t1=float(sample_times[-1]), dt0=1e-11,
    saveat=sample_times, max_steps=4000, throw=True,
)
fitted_output = np.asarray(fitted_circuit.port(fitted_solution.ys, "output"))
pulse_error = np.max(np.abs(fitted_output - np.asarray(analytic_out)))
print(f"Maximum fitted pulse error: {pulse_error:.3e} V")
assert pulse_error < 0.003

fig, (ax_wave, ax_error) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax_wave.plot(sample_times * 1e9, analytic_out, "k--", label="clean line")
ax_wave.plot(sample_times * 1e9, fitted_output.real, label="fit from noisy S data")
ax_wave.set(ylabel="Output voltage (V)")
ax_wave.legend()
ax_error.plot(sample_times * 1e9, (fitted_output - np.asarray(analytic_out)).real * 1e3)
ax_error.set(xlabel="Time (ns)", ylabel="Error (mV)")
plt.tight_layout()
plt.show()


The fit recovers the travel time from noisy training data and uses fewer rational poles. The clean core is static; any retained poles describe what the numerical fit kept, not additional physical modes of the line. We haven't measured a runtime improvement here. The fitted core, extra line variables, and delay-history storage all contribute to simulation cost.

The delay bound and reciprocal, nearly matched structure make this a suitable example for automatic inference. Resonant or strongly reflecting data need more care; see the [fitting API](../../docs/fitting_api.md) for the selection checks.
